In [3]:
import time
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from scipy.optimize import minimize

# ============================================================
# 0) PARAMETERS
# ============================================================
ANN_FAC = 252

# Score weights (dual standardization)
W_SHARPE = 0.7
W_ALPHA = 0.3
LAMBDA_BETA = 0.2

# Sector constraints
ATTACK_SECTORS = {"Semiconductors", "Industrials", "Energy"}
ATTACK_TARGET = 0.70

# Portfolio constraints
MIN_W = 0.05
MAX_PER_SECTOR = 2
N_SELECT = 10

BENCHMARK = "SPY"

# ============================================================
# 1) FIXED BACKTEST DATE (2026 VERSION)
# ============================================================
AS_OF_DATE = datetime(2026, 3, 5)

# IMPORTANT: yfinance end is exclusive → add 1 day
END_3M = AS_OF_DATE + timedelta(days=1)
START_3M = AS_OF_DATE - timedelta(days=92)

END_10Y = AS_OF_DATE + timedelta(days=1)
START_10Y = AS_OF_DATE - timedelta(days=365 * 10 + 10)
# ============================================================
# 2) UNIVERSE
# ============================================================
universe = {
    "Semiconductors": ["NVDA", "TSM", "ASML", "AVGO"],
    "Industrials": ["CAT", "RTX", "GE", "MMM"],
    "Energy": ["XOM", "CVX", "COP", "SHEL"],
    "Materials": ["BHP", "LIN", "NEM", "APD"],
    "ConsumerStaples": ["PG", "PEP", "WMT", "COST"],
    "Healthcare": ["JNJ", "ABBV", "LLY", "UNH"],
}

sector_of = {t: sec for sec, lst in universe.items() for t in lst}
all_candidates = sorted(sector_of.keys())

# ============================================================
# 3) HELPER FUNCTIONS
# ============================================================
def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

def safe_download_close(tickers, start, end):
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )

    if isinstance(data.columns, pd.MultiIndex):
        px = data["Close"]
    else:
        px = data["Close"]

    return px.dropna(how="all").dropna(axis=1, how="all")

def get_rf():
    fred = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DTB3")
    fred.iloc[:, 0] = pd.to_datetime(fred.iloc[:, 0])
    fred.iloc[:, 1] = pd.to_numeric(fred.iloc[:, 1], errors="coerce")

    samp = fred[
        (fred.iloc[:, 0] >= START_10Y) &
        (fred.iloc[:, 0] <= END_10Y)
    ]

    return samp.iloc[:, 1].mean() / 100

# ============================================================
# 4) STEP 1: SCORING (3-MONTH)
# ============================================================
download_list = sorted(set(all_candidates + [BENCHMARK]))

px_3m = safe_download_close(download_list, START_3M, END_3M)
ret_3m = np.log(px_3m).diff().dropna()

rm = ret_3m[BENCHMARK]

rows = []
for t in all_candidates:
    if t not in ret_3m.columns:
        continue

    df = pd.concat([ret_3m[t], rm], axis=1).dropna()
    df.columns = ["Ri", "Rm"]

    if len(df) < 30:
        continue

    sharpe = (df["Ri"].mean() / df["Ri"].std()) * np.sqrt(ANN_FAC)

    X = np.vstack([np.ones(len(df)), df["Rm"]]).T
    y = df["Ri"].values
    alpha, beta = np.linalg.lstsq(X, y, rcond=None)[0]

    rows.append({
        "Ticker": t,
        "Sector": sector_of[t],
        "Sharpe": sharpe,
        "Alpha": alpha * ANN_FAC,
        "Beta": beta
    })

score_df = pd.DataFrame(rows)

# Standardization
score_df["zS"] = zscore(score_df["Sharpe"])
score_df["zA"] = zscore(score_df["Alpha"])
score_df["zB"] = zscore(abs(score_df["Beta"]))

score_df["Score"] = (
    W_SHARPE * score_df["zS"]
    + W_ALPHA * score_df["zA"]
    - LAMBDA_BETA * score_df["zB"]
)

score_df = score_df.sort_values("Score", ascending=False)

# ============================================================
# 5) STEP 2: SELECT STOCKS
# ============================================================
selected = []
sector_count = {}

def can_add(t, s):
    return sector_count.get(s, 0) < MAX_PER_SECTOR and t not in selected

# Prioritize attack sectors
for _, r in score_df.iterrows():
    if len(selected) >= 6:
        break
    if r["Sector"] in ATTACK_SECTORS and can_add(r["Ticker"], r["Sector"]):
        selected.append(r["Ticker"])
        sector_count[r["Sector"]] = sector_count.get(r["Sector"], 0) + 1

# Fill remaining
for _, r in score_df.iterrows():
    if len(selected) >= N_SELECT:
        break
    if can_add(r["Ticker"], r["Sector"]):
        selected.append(r["Ticker"])
        sector_count[r["Sector"]] = sector_count.get(r["Sector"], 0) + 1

# Build final selected dataframe with metrics
sel_df = score_df[score_df["Ticker"].isin(selected)].copy()

# Keep original ranking order
sel_df = sel_df.set_index("Ticker").loc[selected].reset_index()

print("\n=== Selected 10 stocks by Score (sector<=2; attack=Semis/Industrials/Energy prioritized) ===")

print(
    sel_df[
        ["Sector", "Ticker", "Score", "Sharpe", "Alpha", "Beta"]
    ].to_string(index=False)
)

# ============================================================
# 6) STEP 3: MARKOWITZ OPTIMIZATION
# ============================================================
px_10y = safe_download_close(selected, START_10Y, END_10Y)
ret_10y = np.log(px_10y).diff().dropna()

mu = ret_10y.mean().values * ANN_FAC
Sigma = ret_10y.cov().values * ANN_FAC
rf = get_rf()

cols = px_10y.columns
n = len(cols)

is_attack = np.array([sector_of[t] in ATTACK_SECTORS for t in cols])

def sharpe(w):
    r = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    return (r - rf) / vol

def objective(w):
    return -sharpe(w)

constraints = [
    {"type": "eq", "fun": lambda w: np.sum(w) - 1},
    {"type": "eq", "fun": lambda w: w @ is_attack - ATTACK_TARGET}
]

bounds = [(MIN_W, 1) for _ in range(n)]

w0 = np.ones(n) / n

res = minimize(objective, w0, method="SLSQP", bounds=bounds, constraints=constraints)

w = res.x

# ============================================================
# 7) OUTPUT
# ============================================================
result = pd.DataFrame({
    "Ticker": cols,
    "Weight": w,
    "Sector": [sector_of[t] for t in cols]
}).sort_values("Weight", ascending=False)

print("\n=== Final Portfolio ===")
print(result)

print("\nSharpe:", sharpe(w))


=== Selected 10 stocks by Score (sector<=2; attack=Semis/Industrials/Energy prioritized) ===
         Sector Ticker     Score   Sharpe    Alpha      Beta
         Energy    CVX  1.601711 4.134205 0.955705  0.097697
         Energy    XOM  1.555939 3.893734 1.029861  0.041441
    Industrials    RTX  0.970968 3.208138 0.791083  0.628310
 Semiconductors    TSM  0.127181 2.124977 0.751171  2.067652
    Industrials    CAT  0.029335 1.991653 0.736659  2.196028
 Semiconductors   ASML -0.109136 1.767281 0.752300  2.404299
      Materials    LIN  1.281227 3.789103 0.734259  0.247203
     Healthcare    JNJ  1.178254 3.640345 0.648781 -0.191636
      Materials    BHP  0.993989 3.193850 1.086966  1.247730
ConsumerStaples    PEP  0.136208 1.667168 0.371650 -0.262189

=== Final Portfolio ===
  Ticker    Weight           Sector
8    TSM  0.300939   Semiconductors
2    CAT  0.199061      Industrials
4    JNJ  0.146825       Healthcare
5    LIN  0.053175        Materials
0   ASML  0.050000   Semicondu